In [1]:
"""
================================================================================
McNemar's TEST: CRYPTOGUARD-MLP vs CATBOOST
5-Fold CV - Fold-Wise + Pooled Results (3 Output Files)
================================================================================
"""

# ============ SYSTEM & UTILITIES ============
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============ DEEP LEARNING (MLP) ============
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============ SKLEARN - DATA & METRICS ============
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, matthews_corrcoef
from scipy.stats import chi2

# ============ RANDOM SEED & ENVIRONMENT ============
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import random
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

# ============ CONFIG ============
DATASET = "D"
DATE = "14082026"
CV_FOLDS = 5
EPOCHS_FINAL = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

# ============ INPUT FILE (Part 1 Results) ============
FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

# ============ OUTPUT DIRECTORY & FILES ============
OUT_DIR = f"./3. CryptoGuard-MLP_vs_CatBoost_McNemar_Pooled_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)

FILE_INDIVIDUAL_PREDICTIONS = f"{OUT_DIR}/3.1_Individual_Predictions_All_Folds_{DATE}.csv"
FILE_FOLDWISE_CONTINGENCY = f"{OUT_DIR}/3.2_FoldWise_Contingency_Table_{DATE}.csv"
FILE_POOLED_RESULTS = f"{OUT_DIR}/3.3_Pooled_McNemar_Results_{DATE}.csv"

# ============ HELPER FUNCTIONS ============
def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    """Create CryptoGuard-MLP with specified architecture"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer2, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer3, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(l2_reg))
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    """Get training callbacks for early stopping and LR reduction"""
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, 
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

def calculate_metrics(y_true, y_pred):
    """Calculate accuracy and MCC from predictions"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    return acc, mcc

def perform_mcnemar_test_with_contingency(mlp_correct, cat_correct):
    """
    Perform McNemar's test on paired predictions
    
    Contingency Table:
                      CatBoost Correct | CatBoost Incorrect
    MLP Correct              a         |         b
    MLP Incorrect            c         |         d
    
    Returns: (a, b, c, d, chi2_stat, p_value)
    """
    a = np.sum((mlp_correct == 1) & (cat_correct == 1))
    b = np.sum((mlp_correct == 1) & (cat_correct == 0))
    c = np.sum((mlp_correct == 0) & (cat_correct == 1))
    d = np.sum((mlp_correct == 0) & (cat_correct == 0))
    
    # McNemar's test statistic: χ² = (b - c)² / (b + c)
    if (b + c) > 0:
        mcnemar_stat = (b - c) ** 2 / (b + c)
        p_value = 1 - chi2.cdf(mcnemar_stat, df=1)
    else:
        mcnemar_stat = np.nan
        p_value = np.nan
    
    return a, b, c, d, mcnemar_stat, p_value

def print_contingency_table(label, a, b, c, d):
    """Pretty print contingency table for console output"""
    print(f"\n┌─────────────────────────────────────────────────────────────────┐")
    print(f"│          Contingency Table: {label:30s}│")
    print(f"├─────────────────────────────────────────────────────────────────┤")
    print(f"│                                                                 │")
    print(f"│                   CatBoost Correct | CatBoost Incorrect        │")
    print(f"│  ────────────────────────────────────────────────────────────│")
    print(f"│  CryptoGuard Correct   │      {a:6d}       │      {b:6d}       │")
    print(f"│  CryptoGuard Incorrect │      {c:6d}       │      {d:6d}       │")
    print(f"│                                                                 │")
    print(f"└─────────────────────────────────────────────────────────────────┘")

# ============ MAIN EXECUTION ============
if __name__ == "__main__":
    print_header("McNemar's TEST: CRYPTOGUARD-MLP vs CATBOOST (3 OUTPUT FILES)")

    # ============ STEP 1: LOAD GRIDSEARCH RESULTS ============
    print_step(1, "LOADING GRIDSEARCH RESULTS")
    
    if not os.path.exists(FILE_RESULTS):
        print(f"❌ Error: {FILE_RESULTS} not found")
        sys.exit(1)
    
    df_results = pd.read_csv(FILE_RESULTS)
    best_row = df_results.iloc[0]
    
    h1 = int(best_row['param_hidden_layer1'])
    h2 = int(best_row['param_hidden_layer2'])
    h3 = int(best_row['param_hidden_layer3'])
    dr = float(best_row['param_dropout_rate'])
    l2r = float(best_row['param_l2_reg'])
    lr = float(best_row['param_learning_rate'])
    bs = int(best_row['param_batch_size'])
    
    print(f"\n✓ Best MLP Configuration (Locked):")
    print(f"  Architecture: {h1}-{h2}-{h3}")
    print(f"  Dropout: {dr} | L2: {l2r} | LR: {lr} | BS: {bs}")

    # ============ STEP 2: LOAD DATASET ============
    print_step(2, "LOADING DATASET")
    
    df = pd.read_excel(f'./Datasets/{DATASET}.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    
    print(f"\n✓ Dataset: {DATASET}")
    print(f"  Total Samples: {len(df):,}")
    print(f"  Features: {input_dim}")
    print(f"  Folds: {CV_FOLDS} | Test per fold: {len(df) // CV_FOLDS:,}")

    # ============ STEP 3: 5-FOLD CV ============
    print_step(3, "5-FOLD STRATIFIED CROSS-VALIDATION")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    all_predictions = []
    all_mlp_correct = []
    all_cat_correct = []
    foldwise_contingency = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n  [Fold {fold}/{CV_FOLDS}]", end='', flush=True)
        
        # ========== Data Split ==========
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx].values.astype(np.float32)
        y_test = y.iloc[test_idx].values.astype(np.float32)
        
        # ========== Validation Split (10% of training) ==========
        X_train_real, X_val, y_train_real, y_val = train_test_split(
            X_train, y_train,
            test_size=0.10,
            random_state=SEED,
            stratify=y_train
        )
        
        # ========== Feature Scaling ==========
        scaler = StandardScaler()
        X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        X_test_sc = scaler.transform(X_test).astype(np.float32)
        
        # ========== MODEL 1: CRYPTOGUARD-MLP ==========
        print(f" MLP", end='', flush=True)
        model_mlp = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
        history_mlp = model_mlp.fit(
            X_train_real_sc, y_train_real,
            epochs=EPOCHS_FINAL,
            batch_size=bs,
            validation_data=(X_val_sc, y_val),
            callbacks=get_callbacks(),
            verbose=0
        )
        
        y_pred_mlp_proba = model_mlp.predict(X_test_sc, verbose=0, batch_size=bs)
        y_pred_mlp = (y_pred_mlp_proba > 0.5).astype(int).flatten()
        mlp_acc, mlp_mcc = calculate_metrics(y_test, y_pred_mlp)
        
        print(f"({mlp_acc*100:.2f}%)", end='', flush=True)
        
        del model_mlp, history_mlp
        tf.keras.backend.clear_session()
        gc.collect()
        
        # ========== MODEL 2: CATBOOST ==========
        print(f" | CatBoost", end='', flush=True)
        cat_model = CatBoostClassifier(
            iterations=100,
            learning_rate=0.01,
            depth=6,
            early_stopping_rounds=PATIENCE_ES,
            random_state=SEED,
            verbose=0
        )
        cat_model.fit(
            X_train_real_sc, y_train_real,
            eval_set=(X_val_sc, y_val),
            verbose=0
        )
        
        y_pred_cat = cat_model.predict(X_test_sc)
        cat_acc, cat_mcc = calculate_metrics(y_test, y_pred_cat)
        
        print(f"({cat_acc*100:.2f}%)", end='', flush=True)
        
        del cat_model
        gc.collect()
        
        # ========== STORE PREDICTIONS & COMPUTE FOLD CONTINGENCY ==========
        fold_mlp_correct = []
        fold_cat_correct = []
        
        for i, (true_label, mlp_pred, cat_pred) in enumerate(zip(y_test, y_pred_mlp, y_pred_cat)):
            mlp_is_correct = int(true_label == mlp_pred)
            cat_is_correct = int(true_label == cat_pred)
            
            fold_mlp_correct.append(mlp_is_correct)
            fold_cat_correct.append(cat_is_correct)
            all_mlp_correct.append(mlp_is_correct)
            all_cat_correct.append(cat_is_correct)
            
            all_predictions.append({
                'Fold': fold,
                'Sample_ID': i,
                'True_Label': int(true_label),
                'MLP_Prediction': int(mlp_pred),
                'CatBoost_Prediction': int(cat_pred),
                'MLP_Correct': mlp_is_correct,
                'CatBoost_Correct': cat_is_correct
            })
        
        # ========== FOLD CONTINGENCY TABLE ==========
        fold_a, fold_b, fold_c, fold_d, _, _ = perform_mcnemar_test_with_contingency(
            np.array(fold_mlp_correct), 
            np.array(fold_cat_correct)
        )
        
        foldwise_contingency.append({
            'Fold': fold,
            'Total_Samples': len(y_test),
            'a_Both_Correct': fold_a,
            'b_MLP_Advantage': fold_b,
            'c_CryptoGuard_Disadvantage': fold_c,
            'd_Both_Wrong': fold_d
        })
        
        print(f" ✓")
        
        del X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc
        gc.collect()

    # ============ STEP 4: SAVE FILE 1 - INDIVIDUAL PREDICTIONS ============
    print_step(4, "SAVING FILE 1: INDIVIDUAL PREDICTIONS")
    
    predictions_df = pd.DataFrame(all_predictions)
    predictions_df.to_csv(FILE_INDIVIDUAL_PREDICTIONS, index=False)
    print(f"\n✓ Saved: {FILE_INDIVIDUAL_PREDICTIONS}")
    print(f"  └─ {len(predictions_df):,} predictions ({CV_FOLDS} folds × {len(predictions_df)//CV_FOLDS:,} samples/fold)")

    # ============ STEP 5: SAVE FILE 2 - FOLD-WISE CONTINGENCY ============
    print_step(5, "SAVING FILE 2: FOLD-WISE CONTINGENCY TABLE")
    
    foldwise_df = pd.DataFrame(foldwise_contingency)
    foldwise_df.to_csv(FILE_FOLDWISE_CONTINGENCY, index=False)
    print(f"\n✓ Saved: {FILE_FOLDWISE_CONTINGENCY}")
    print(f"\n{foldwise_df.to_string(index=False)}")

    # ============ STEP 6: PERFORM POOLED McNemar's TEST ============
    print_step(6, "PERFORMING McNemar's TEST (POOLED - ALL FOLDS)")
    
    pooled_a, pooled_b, pooled_c, pooled_d, pooled_stat, pooled_p = perform_mcnemar_test_with_contingency(
        np.array(all_mlp_correct), 
        np.array(all_cat_correct)
    )
    
    # Print contingency table
    print_contingency_table("POOLED (ALL FOLDS)", pooled_a, pooled_b, pooled_c, pooled_d)
    
    # Print detailed statistics
    print(f"\n✓ McNemar's Test Calculation:")
    print(f"  Total Samples: {len(all_mlp_correct):,}")
    print(f"  a (Both Correct): {pooled_a:,}")
    print(f"  b (MLP Advantage): {pooled_b:,}")
    print(f"  c (CryptoGuard Disadvantage): {pooled_c:,}")
    print(f"  d (Both Wrong): {pooled_d:,}")
    print(f"  ─────────────────────────────────────────")
    print(f"  Formula: χ² = (b - c)² / (b + c)")
    print(f"  χ² = ({pooled_b} - {pooled_c})² / ({pooled_b} + {pooled_c})")
    print(f"  χ² = {(pooled_b - pooled_c)**2} / {pooled_b + pooled_c}")
    print(f"  χ² = {pooled_stat:.4f}")
    print(f"  P-value: {pooled_p:.10f}")
    print(f"  Effect Size (b/c ratio): {pooled_b/pooled_c:.4f}")
    
    # Significance interpretation
    print(f"\n✓ RESULT: ", end='')
    if pooled_p < 0.05:
        print("✓ SIGNIFICANT (p < 0.05)")
        print(f"  └─ CryptoGuard-MLP significantly outperforms CatBoost")
    else:
        print("✗ NOT SIGNIFICANT (p ≥ 0.05)")

    # ============ STEP 7: SAVE FILE 3 - POOLED RESULTS ============
    print_step(7, "SAVING FILE 3: POOLED McNemar's TEST RESULTS")
    
    pooled_results = pd.DataFrame({
        'Test': ['McNemar\'s Test (Pooled)'],
        'Model_1': ['CryptoGuard-MLP'],
        'Model_2': ['CatBoost'],
        'Total_Samples': [len(all_mlp_correct)],
        'Total_Folds': [CV_FOLDS],
        'a_Both_Correct': [pooled_a],
        'b_MLP_Advantage': [pooled_b],
        'c_CryptoGuard_Disadvantage': [pooled_c],
        'd_Both_Wrong': [pooled_d],
        'Chi_Square_Statistic': [f"{pooled_stat:.4f}"],
        'P_Value': [f"{pooled_p:.10f}"],
        'Significant_at_Alpha_0.05': ['Yes' if pooled_p < 0.05 else 'No'],
        'Effect_Size_b_c_Ratio': [f"{pooled_b/pooled_c:.4f}"]
    })
    
    pooled_results.to_csv(FILE_POOLED_RESULTS, index=False)
    print(f"\n✓ Saved: {FILE_POOLED_RESULTS}")
    print(f"\n{pooled_results.to_string(index=False)}")

    # ============ FINAL SUMMARY ============
    print("\n" + "="*80)
    print("✓ McNemar's TEST COMPLETE - 3 FILES GENERATED!")
    print("="*80)
    
    print(f"\nOutput Directory: {OUT_DIR}\n")
    print(f"Generated Files:")
    print(f"  ✓ 3.1_Individual_Predictions_All_Folds_{DATE}.csv")
    print(f"      Columns: Fold, Sample_ID, True_Label, MLP_Prediction, CatBoost_Prediction,")
    print(f"               MLP_Correct, CatBoost_Correct")
    print(f"      Rows: {len(predictions_df):,} (20k total predictions)")
    
    print(f"\n  ✓ 3.2_FoldWise_Contingency_Table_{DATE}.csv")
    print(f"      Columns: Fold, Total_Samples, a_Both_Correct, b_MLP_Advantage,")
    print(f"               c_CryptoGuard_Disadvantage, d_Both_Wrong")
    print(f"      Rows: {len(foldwise_df)} (one per fold)")
    
    print(f"\n  ✓ 3.3_Pooled_McNemar_Results_{DATE}.csv")
    print(f"      Columns: Test, Model_1, Model_2, Total_Samples, Total_Folds,")
    print(f"               a, b, c, d, Chi_Square_Statistic, P_Value, Significant, Effect_Size")
    print(f"      Rows: 1 (pooled across all folds)")
    
    print(f"\n{'─'*80}")
    print(f"Pooled McNemar's Test Summary:")
    print(f"  χ² = {pooled_stat:.4f}")
    print(f"  p-value = {pooled_p:.10f}")
    print(f"  Significant (α=0.05): {'Yes ✓' if pooled_p < 0.05 else 'No ✗'}")
    print(f"  Effect Size (b/c): {pooled_b/pooled_c:.4f}")
    print(f"{'─'*80}")
    
    print("\n✓ Ready for LaTeX table and paper writing!")
    print("="*80)


McNemar's TEST: CRYPTOGUARD-MLP vs CATBOOST (3 OUTPUT FILES)

[STEP 1] LOADING GRIDSEARCH RESULTS
--------------------------------------------------------------------------------

✓ Best MLP Configuration (Locked):
  Architecture: 768-512-128
  Dropout: 0.5 | L2: 0.005 | LR: 0.0001 | BS: 64

[STEP 2] LOADING DATASET
--------------------------------------------------------------------------------

✓ Dataset: D
  Total Samples: 20,000
  Features: 525
  Folds: 5 | Test per fold: 4,000

[STEP 3] 5-FOLD STRATIFIED CROSS-VALIDATION
--------------------------------------------------------------------------------

  [Fold 1/5] MLP(93.03%) | CatBoost(92.10%) ✓

  [Fold 2/5] MLP(93.12%) | CatBoost(92.50%) ✓

  [Fold 3/5] MLP(92.73%) | CatBoost(91.27%) ✓

  [Fold 4/5] MLP(93.62%) | CatBoost(92.42%) ✓

  [Fold 5/5] MLP(92.77%) | CatBoost(92.05%) ✓

[STEP 4] SAVING FILE 1: INDIVIDUAL PREDICTIONS
--------------------------------------------------------------------------------

✓ Saved: ./3. CryptoG

In [2]:
import numpy as np
from scipy.stats import chi2

# Your data
a, b, c, d = 17875, 736, 539, 850

# McNemar's test
chi_square_stat = (b - c) ** 2 / (b + c)
p_value = 1 - chi2.cdf(chi_square_stat, df=1)
effect_size = b / c

print(f"χ² = {chi_square_stat:.4f}")
print(f"p = {p_value:.10f}")
print(f"Significant: {p_value < 0.05}")
print(f"Effect Size (b/c): {effect_size:.4f}")

# Output:
# χ² = 30.4384
# p = 0.0000000345
# Significant: True
# Effect Size (b/c): 1.3655

χ² = 30.4384
p = 0.0000000345
Significant: True
Effect Size (b/c): 1.3655
